In [1]:
import pandas as pd
import os
import glob
import re

In [9]:
paths = {
    'Rating': r'C:\Users\PAOLA\Desktop\Proyectos personales\Rendimiento_vs_salario\Rating\Limpios',
    'Rendimiento': r'C:\Users\PAOLA\Desktop\Proyectos personales\Rendimiento_vs_salario\Rendimiento',
    'Salarios': r'C:\Users\PAOLA\Desktop\Proyectos personales\Rendimiento_vs_salario\Salarios'
}
folder_output = r'C:\Users\PAOLA\Desktop\Proyectos personales\Rendimiento_vs_salario\Dashboard_Data'
ligas = ['La_liga', 'Premier', 'Bundesliga', 'Serie_a', 'Ligue']

def ejecutar_limpieza_v8():
    data_consolidada = {}

    for categoria, ruta_base in paths.items():
        print(f"\n🚀 PROCESANDO CATEGORÍA: {categoria}")
        acumulado = []
        
        for liga in ligas:
            ruta_liga = os.path.join(ruta_base, liga)
            if not os.path.exists(ruta_liga): continue
            
            archivos = glob.glob(os.path.join(ruta_liga, "*.csv")) + glob.glob(os.path.join(ruta_liga, "*.xlsx"))
            
            for archivo in archivos:
                try:
                    if archivo.endswith('.csv'):
                        # Capology a veces tiene codificación latin-1 o filas iniciales raras
                        df = pd.read_csv(archivo, on_bad_lines='skip', engine='python', encoding='utf-8-sig')
                    else:
                        df = pd.read_excel(archivo, engine='openpyxl')
                    
                    # Limpieza básica de columnas
                    df.columns = [c.lower().strip().replace(' ', '_') for c in df.columns]
                    
                    # --- NUEVA LÓGICA DE BÚSQUEDA DE JUGADOR (MÁS AGRESIVA) ---
                    # Si no encuentra las palabras clave, buscamos cualquier columna que contenga 'player' o 'name'
                    col_jugador = None
                    posibles = ['player', 'player_name', 'name', 'nombre', 'full_name', 'jugador', 'athlete']
                    
                    # 1. Intento por coincidencia exacta
                    for p in posibles:
                        if p in df.columns:
                            col_jugador = p
                            break
                    
                    # 2. Intento por coincidencia parcial (si falla el 1)
                    if not col_jugador:
                        for c in df.columns:
                            if 'player' in c or 'name' in c:
                                col_jugador = c
                                break
                    
                    if col_jugador:
                        df.rename(columns={col_jugador: 'player'}, inplace=True)
                    else:
                        # Si es Salarios, imprimimos las columnas para saber qué pasa
                        if categoria == 'Salarios':
                            print(f"    ❓ Columnas no reconocidas en {os.path.basename(archivo)}: {list(df.columns[:4])}")
                        continue

                    # Temporada
                    nombre_archivo = os.path.basename(archivo).lower()
                    numeros = re.findall(r'\d+', nombre_archivo)
                    anio_base = 2023
                    for n in numeros:
                        v = int(n)
                        if 2020 <= v <= 2026: anio_base = v; break
                        if 20 <= v <= 26: anio_base = 2000 + v; break
                    
                    df['temporada'] = f"{anio_base}-{anio_base + 1}"
                    df['liga'] = liga

                    # Limpieza Dinero
                    cols_money = [c for c in df.columns if 'wage' in c or 'salary' in c or 'base_pay' in c]
                    for c in cols_money:
                        df[c] = df[c].astype(str).str.replace(r'[€,£$,]', '', regex=True)
                        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)

                    acumulado.append(df)
                    
                except Exception as e:
                    print(f"    ❌ Error en {os.path.basename(archivo)}: {e}")
        
        if acumulado:
            df_final = pd.concat(acumulado, ignore_index=True)
            print(f"  ✅ {categoria} listo con {len(df_final)} filas.")
            
            if not os.path.exists(folder_output): os.makedirs(folder_output)
            path_output = os.path.join(folder_output, f'Master_{categoria}.csv')
            df_final.to_csv(path_output, index=False, encoding='utf-8-sig')
            print(f"  💾 GUARDADO: {path_output}")
        else:
            print(f"  🛑 ATENCIÓN: No se generaron datos para {categoria}. Revisa los nombres de las columnas arriba.")

    return data_consolidada

tablas = ejecutar_limpieza_v8()


🚀 PROCESANDO CATEGORÍA: Rating
  ✅ Rating listo con 8608 filas.
  💾 GUARDADO: C:\Users\PAOLA\Desktop\Proyectos personales\Rendimiento_vs_salario\Dashboard_Data\Master_Rating.csv

🚀 PROCESANDO CATEGORÍA: Rendimiento
  ✅ Rendimiento listo con 10209 filas.
  💾 GUARDADO: C:\Users\PAOLA\Desktop\Proyectos personales\Rendimiento_vs_salario\Dashboard_Data\Master_Rendimiento.csv

🚀 PROCESANDO CATEGORÍA: Salarios
  ✅ Salarios listo con 10878 filas.
  💾 GUARDADO: C:\Users\PAOLA\Desktop\Proyectos personales\Rendimiento_vs_salario\Dashboard_Data\Master_Salarios.csv
